In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

%matplotlib inline

In [ ]:
root = Path('..')
data_path = root / 'data' / 'names.txt'
words = open(data_path).read().splitlines()

print(f'Words count: {len(words)}')

In [ ]:
# build vocabulary of characters and mappings to/from ints

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

In [ ]:
# build dataset (batch of 5 words)

block_size = 3
X, Y = [], []

for w in words[:5]:

    print(w)
    context = [0] * block_size # padded context
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(f"{''.join(itos[i] for i in context)} ---> {itos[ix]}")
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

print(X.shape, X.dtype, Y.shape, Y.dtype)

In [ ]:
g = torch.Generator().manual_seed(42)
C = torch.randn((27, 2)) # 27 characters, 2 dimensions
print(C.shape)

W1 = torch.randn((6, 100))  # transforms flattened 6 features to 100 hidden units
b1 = torch.randn(100)       # bias term for the hidden layer
W2 = torch.randn((100, 27)) # transforms 100 hidden units to 27 output units
b2 = torch.randn(27)        # bias term for the output layer

parameters = [C, W1, b1, W2, b2]

In [ ]:
sum(p.nelement() for p in parameters) # total number of parameters

In [ ]:
emb = C[X]
print(emb.shape)

hidden_layer = torch.tanh(emb.view(-1, 6) @ W1 + b1)
print(hidden_layer.shape)

logits = hidden_layer @ W2 + b2
print(logits.shape)

counts = logits.exp()
probs = counts / counts.sum(-1, keepdim=True)
print(probs.shape)

loss = -probs[torch.arange(len(Y)), Y].log().mean()
print(loss)



In [ ]:
# build dataset (all words)

block_size = 3
X, Y = [], []

for w in words:

    context = [0] * block_size # padded context
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

print(X.shape, X.dtype, Y.shape, Y.dtype)

In [ ]:
g = torch.Generator().manual_seed(42)
C = torch.randn((27, 2)) # 27 characters, 2 dimensions
print(C.shape)

W1 = torch.randn((6, 100))  # transforms flattened 6 features to 100 hidden units
b1 = torch.randn(100)       # bias term for the hidden layer
W2 = torch.randn((100, 27)) # transforms 100 hidden units to 27 output units
b2 = torch.randn(27)        # bias term for the output layer

parameters = [C, W1, b1, W2, b2]

sum(p.nelement() for p in parameters) # total number of parameters

for p in parameters: p.requires_grad = True # enable gradients

In [ ]:
# Training loop
# use cross-entropy loss

EPOCHS = 10_000
BATCH_SIZE = 64

LR = 0.13974539020061493
# LRE = torch.linspace(-3, 0, EPOCHS)
# LRS = 10**LRE # create logarithmically spaced learning rates

lri = []
losses = []

for epoch in range(EPOCHS):

    # minibatch
    # Step 1 - sample a random minibatch of BATCH_SIZE examples
    indices = torch.randint(0, len(X), (BATCH_SIZE,), generator=g)

    # forward pass
    emb = C[X[indices]] # Grab the embeddings for the current minibatch
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[indices])
    if epoch % 1000 == 0: print(f'epoch: {epoch} loss: {loss.item()}')

    # backward pass
    for p in parameters: p.grad = None
    loss.backward()

    # gradient descent step 
    # lr = LRS[epoch]
    lr = LR
    for p in parameters: p.data -= p.grad * lr

    # track loss
    # lri.append(lr)
    # losses.append(loss.item())

In [ ]:
# min_loss_idx = losses.index(min(losses))
# optimal_lr = lri[min_loss_idx]
# print(f'Optimal LR: {optimal_lr}')

In [ ]:
# plt.plot(lri, losses)

In [ ]:
# evaluate on the whole dataset
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
print(f'final loss: {loss.item()}')

In [ ]:
# build dataset split into train / val / test

def build_dataset(words, block_size=3):
    X, Y = [], []

    for w in words:

        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, X.dtype, Y.shape, Y.dtype)
    return X, Y

X, Y = build_dataset(words)

s1 = int(0.8 * len(X))
s2 = int(0.9 * len(X))

X_train, Y_train = X[:s1], Y[:s1]
X_val, Y_val = X[s1:s2], Y[s1:s2]
X_test, Y_test = X[s2:], Y[s2:]

print(f'train: {len(X_train)} val: {len(X_val)} test: {len(X_test)}')

In [ ]:
g = torch.Generator().manual_seed(42)
C = torch.randn((27, 10)) # 27 characters, increased embedding size to 10 dimensions
print(C.shape)

W1 = torch.randn((30, 1000)) # increased hidden units to 1000, features to 30 (10 * 3)
b1 = torch.randn(1000)
W2 = torch.randn((1000, 27))
b2 = torch.randn(27)

parameters = [C, W1, b1, W2, b2]

total_params = sum(p.nelement() for p in parameters) # total number of parameters
print(f'Total parameters: {total_params}')

for p in parameters: p.requires_grad = True # enable gradients

In [ ]:
EPOCHS = 250_000
BATCH_SIZE = 32

LR = 0.1

lri = []
lossi = []
stepi = [] # let's also track the training steps

In [ ]:
# Training loop
# use cross-entropy loss

for epoch in range(EPOCHS):

    # minibatch
    indices = torch.randint(0, len(X_train), (BATCH_SIZE,), generator=g)

    # forward pass
    emb = C[X_train[indices]]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y_train[indices])
    if epoch % 25_000 == 0: print(f'epoch: {epoch} loss: {loss.item()}')

    # backward pass
    for p in parameters: p.grad = None
    loss.backward()

    # gradient descent step 
    lr = LR if epoch < 125_000 else LR / 10 # weight decay after 125k steps
    for p in parameters: p.data -= p.grad * lr

    # track loss
    lri.append(lr)
    lossi.append(loss.log10().item())
    stepi.append(epoch)

# evaluate on the whole dataset
emb = C[X_test]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y_test)
print(f'final loss (on test set): {loss.item()}')

In [ ]:
plt.plot(stepi, lossi)

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(C[:, 0].data, C[:, 1].data, s=200)
for i in range(C.shape[0]):
    plt.text(C[i, 0].item(), C[i, 1].item(), itos[i], ha='center', va='center', color='white')
plt.grid('both')

In [ ]:
# sample from the model

block_size = 3
g = torch.Generator().manual_seed(42)

for i in range(25):

    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])] # (1, block_size, dim)
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0: break

    print(''.join(itos[i] for i in out))